In [59]:
import numpy as np
import librosa
import parselmouth

SR = 16000

def extract_behavioral_features(file_path):
    y, sr = librosa.load(file_path, sr=SR, mono=True)
    y = y / max(abs(y))
    
    # ---------------- Voice features ----------------
    snd = parselmouth.Sound(file_path)
    
    # Pitch
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]
    pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_std = np.std(pitch_values) if len(pitch_values) > 0 else 0
    
    # Jitter & Shimmer
    point_process = parselmouth.praat.call(snd, "To PointProcess (periodic, cc)", 75, 500)
    jitter_local = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    shimmer_local = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    
    # Energy
    energy = np.mean(y**2)
    
    # Speaking rate (number of onsets / duration)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    duration_sec = librosa.get_duration(y=y, sr=sr)
    speaking_rate = len(onsets)/dura_


In [60]:
def extract_behavioral_features(file_path):
    y, sr = librosa.load(file_path, sr=SR, mono=True)
    y = y / max(abs(y))
    
    # ---------------- Voice features ----------------
    snd = parselmouth.Sound(file_path)
    
    # Pitch
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]
    pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_std = np.std(pitch_values) if len(pitch_values) > 0 else 0
    
    # Jitter & Shimmer
    point_process = parselmouth.praat.call(snd, "To PointProcess (periodic, cc)", 75, 500)
    jitter_local = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    shimmer_local = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    
    # Energy
    energy = np.mean(y**2)
    
    # Speaking rate (number of onsets / duration)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    duration_sec = librosa.get_duration(y=y, sr=sr)
    speaking_rate = len(onsets)/duration_sec if duration_sec > 0 else 0  # <-- fixed here
    
    # MFCCs (mean over frames)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    mfcc_mean = np.mean(mfccs.T, axis=0)
    
    # Combine all features
    features = np.hstack([mfcc_mean, pitch_mean, pitch_std, jitter_local, shimmer_local, energy, speaking_rate])
    
    return features


In [61]:
import os
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import torch

root_folder = "User_Voice"  # U001-U023

X = []
y = []

for user_folder in sorted(os.listdir(root_folder)):
    user_path = os.path.join(root_folder, user_folder)
    if not os.path.isdir(user_path):
        continue
    for file in sorted(os.listdir(user_path)):
        if file.endswith(".wav"):
            file_path = os.path.join(user_path, file)
            feats = extract_behavioral_features(file_path)
            X.append(feats)
            y.append(user_folder)

X = np.array(X)
y = np.array(y)

# Encode speaker labels
le_user = LabelEncoder()
y_enc = le_user.fit_transform(y)

print("Feature shape:", X.shape)
print("Number of speakers:", len(le_user.classes_))


Feature shape: (230, 26)
Number of speakers: 23


In [62]:
class SpeakerDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = SpeakerDataset(X, y_enc)
train_size = int(0.8*len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


In [63]:
import torch.nn as nn

class SpeakerNet(nn.Module):
    def __init__(self, input_dim, num_speakers):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_speakers)
        )
    
    def forward(self, x):
        return self.net(x)

model = SpeakerNet(X.shape[1], len(le_user.classes_))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)


SpeakerNet(
  (net): Sequential(
    (0): Linear(in_features=26, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=23, bias=True)
  )
)

In [64]:
import torch.optim as optim
from sklearn.metrics import accuracy_score

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")


Epoch 1, Loss: 5.0697
Epoch 2, Loss: 2.7306
Epoch 3, Loss: 2.2931
Epoch 4, Loss: 1.9436
Epoch 5, Loss: 1.7215
Epoch 6, Loss: 1.4066
Epoch 7, Loss: 1.1703
Epoch 8, Loss: 1.0042
Epoch 9, Loss: 0.8632
Epoch 10, Loss: 0.7065
Epoch 11, Loss: 0.5795
Epoch 12, Loss: 0.5376
Epoch 13, Loss: 0.4964
Epoch 14, Loss: 0.4216
Epoch 15, Loss: 0.3350
Epoch 16, Loss: 0.2765
Epoch 17, Loss: 0.2670
Epoch 18, Loss: 0.2110
Epoch 19, Loss: 0.2029
Epoch 20, Loss: 0.1796
Epoch 21, Loss: 0.1473
Epoch 22, Loss: 0.1271
Epoch 23, Loss: 0.1177
Epoch 24, Loss: 0.1080
Epoch 25, Loss: 0.0985
Epoch 26, Loss: 0.0962
Epoch 27, Loss: 0.0828
Epoch 28, Loss: 0.0796
Epoch 29, Loss: 0.0655
Epoch 30, Loss: 0.0568


In [65]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        outputs = model(x_batch)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

from sklearn.metrics import accuracy_score
print("Speaker identification accuracy:", accuracy_score(all_labels, all_preds))


Speaker identification accuracy: 0.9565217391304348


In [66]:
import pandas as pd

# Compute average behavioral features per speaker
speaker_profiles = {}
for user_folder in sorted(os.listdir(root_folder)):
    user_path = os.path.join(root_folder, user_folder)
    feats_list = []
    if not os.path.isdir(user_path):
        continue
    for file in os.listdir(user_path):
        if file.endswith(".wav"):
            file_path = os.path.join(user_path, file)
            feats = extract_behavioral_features(file_path)
            feats_list.append(feats)
    speaker_profiles[user_folder] = np.mean(feats_list, axis=0)

# Convert to DataFrame
df_profiles = pd.DataFrame.from_dict(speaker_profiles, orient='index')
df_profiles.to_csv("behavioral_voiceprints.csv")
print("Saved behavioral profiles for each speaker.")


Saved behavioral profiles for each speaker.


In [67]:
def identify_speaker(file_path):
    model.eval()
    feats = extract_behavioral_features(file_path)
    x_tensor = torch.tensor(feats, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(x_tensor)
        pred_idx = out.argmax(dim=1).cpu().item()
        speaker = le_user.inverse_transform([pred_idx])[0]
    return speaker, feats

# Example
file = "User_Voice/U004/01.wav"
speaker, behavioral_feats = identify_speaker(file)
print(f"Identified Speaker: {speaker}")
print("Behavioral Features:", behavioral_feats)


Identified Speaker: U004
Behavioral Features: [-2.18139633e+02  1.18982727e+02  2.09231033e+01  1.16857159e+00
  1.49260035e+01 -2.22645817e+01  5.54904079e+00  5.17445421e+00
 -3.16425943e+00  3.41946793e+00  9.39870262e+00  1.27512681e+00
  2.38009644e+00  6.72261763e+00 -1.09468641e+01  4.43326950e+00
 -4.88294554e+00  4.29107046e+00 -1.17683325e+01  4.26520920e+00
  1.78818563e+02  1.23663618e+02  1.77291931e-02  1.21732062e-01
  1.90091655e-02  6.03693182e+00]


In [68]:
torch.save(model.state_dict(), "speaker_model.pth")
print("Model saved successfully.")


Model saved successfully.


In [91]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load speaker profiles
df_profiles = pd.read_csv("behavioral_voiceprints.csv", index_col=0)
print("Loaded behavioral profiles for speakers:", df_profiles.index.tolist())


Loaded behavioral profiles for speakers: ['U001', 'U002', 'U003', 'U004', 'U005', 'U006', 'U007', 'U008', 'U009', 'U010', 'U011', 'U012', 'U013', 'U014', 'U015', 'U016', 'U017', 'U018', 'U019', 'U020', 'U021', 'U022', 'U023']


In [92]:
def verify_speaker(file_path, model, le_user, df_profiles, device='cpu'):
    """
    Inputs:
        file_path : path to new audio
        model     : trained speaker identification model
        le_user   : label encoder for speaker IDs
        df_profiles : DataFrame with stored behavioral profiles
    Returns:
        identified_speaker : predicted speaker ID
        similarity_score   : cosine similarity with stored profile
        behavioral_feats   : feature vector of new sample
    """
    # Extract behavioral features
    behavioral_feats = extract_behavioral_features(file_path)
    
    # Convert to tensor for model prediction
    x_tensor = torch.tensor(behavioral_feats, dtype=torch.float32).unsqueeze(0).to(device)
    
    # Predict speaker
    model.eval()
    with torch.no_grad():
        output = model(x_tensor)
        pred_idx = output.argmax(dim=1).cpu().item()
        identified_speaker = le_user.inverse_transform([pred_idx])[0]
    
    # Compare with stored profile
    stored_profile = df_profiles.loc[identified_speaker].values.reshape(1, -1)
    new_feat = behavioral_feats.reshape(1, -1)
    similarity_score = cosine_similarity(stored_profile, new_feat)[0][0]
    
    return identified_speaker, similarity_score, behavioral_feats


In [93]:
import pandas as pd
import numpy as np

# Load behavioral profiles (average feature vectors per user)
df_profiles = pd.read_csv("behavioral_voiceprints.csv", index_col=0)

print("Loaded profiles for:", df_profiles.index.tolist())
print("Feature dimension:", df_profiles.shape[1])


Loaded profiles for: ['U001', 'U002', 'U003', 'U004', 'U005', 'U006', 'U007', 'U008', 'U009', 'U010', 'U011', 'U012', 'U013', 'U014', 'U015', 'U016', 'U017', 'U018', 'U019', 'U020', 'U021', 'U022', 'U023']
Feature dimension: 26


In [72]:
from sklearn.preprocessing import LabelEncoder
import os

# Recreate label encoder from your existing user folders
root_folder = "User_Voice"
user_folders = sorted([f for f in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, f))])

le_user = LabelEncoder()
le_user.fit(user_folders)

print("Loaded label encoder with users:", le_user.classes_)


Loaded label encoder with users: ['U001' 'U002' 'U003' 'U004' 'U005' 'U006' 'U007' 'U008' 'U009' 'U010'
 'U011' 'U012' 'U013' 'U014' 'U015' 'U016' 'U017' 'U018' 'U019' 'U020'
 'U021' 'U022' 'U023']


In [90]:
import numpy as np
import librosa
import parselmouth

SR = 16000

def extract_behavioral_features(file_path):
    y, sr = librosa.load(file_path, sr=SR, mono=True)
    y = y / max(abs(y))
    
    # ---------------- Voice features ----------------
    snd = parselmouth.Sound(file_path)
    
    # Pitch
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]
    pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_std = np.std(pitch_values) if len(pitch_values) > 0 else 0
    
    # Jitter & Shimmer
    point_process = parselmouth.praat.call(snd, "To PointProcess (periodic, cc)", 75, 500)
    jitter_local = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    shimmer_local = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    
    # Energy
    energy = np.mean(y**2)
    
    # Speaking rate (number of onsets / duration)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    duration_sec = librosa.get_duration(y=y, sr=sr)
    speaking_rate = len(onsets) / duration_sec if duration_sec > 0 else 0
    
    # MFCCs (mean over frames)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    mfcc_mean = np.mean(mfccs.T, axis=0)
    
    # Combine all features
    features = np.hstack([
        mfcc_mean,
        pitch_mean,
        pitch_std,
        jitter_local,
        shimmer_local,
        energy,
        speaking_rate
    ])
    
    return features


In [74]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import librosa
import parselmouth
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity


In [75]:
file = "User_Voice/U007/01.wav"
speaker, similarity, feats = verify_speaker(file, model, le_user, df_profiles, device)

print(f"Identified Speaker: {speaker}")
print(f"Behavioral similarity with stored profile: {similarity:.4f}")
print("Behavioral Features:", feats)


Identified Speaker: U007
Behavioral similarity with stored profile: 0.9942
Behavioral Features: [-2.23262634e+02  1.37446304e+02  2.06232429e+00 -1.69721454e-01
 -1.44573936e+01  5.26486397e+00 -1.33919744e+01  1.93592966e+00
 -8.35824585e+00 -7.89306641e+00  2.00485182e+00 -7.03764248e+00
 -1.40721130e+00 -4.15081739e+00 -6.63053608e+00  2.79008150e+00
 -6.52093792e+00 -5.35275757e-01 -3.23171115e+00 -6.27790451e+00
  1.74385218e+02  1.61413995e+01  1.44655038e-02  8.16930953e-02
  2.88584884e-02  7.26632795e+00]


In [89]:
def analyze_behavioral_features(features):
    mfcc = features[:20]
    pitch_mean = features[20]
    pitch_std = features[21]
    jitter = features[22]
    shimmer = features[23]
    energy = features[24]
    speaking_rate = features[25]
    
    print(f"Pitch mean: {pitch_mean:.2f} Hz, Pitch variability: {pitch_std:.2f} Hz")
    print(f"Jitter: {jitter:.4f}, Shimmer: {shimmer:.4f}")
    print(f"Energy: {energy:.4f}, Speaking rate: {speaking_rate:.2f} onsets/sec")


In [98]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import torch

# ---------- Configuration ----------
SAMPLE_RATE = 16000
DURATION = 5  # seconds
THRESHOLD = 0.95  # similarity threshold for existing users

# ---------- Function to record audio ----------
def record_audio(filename="temp_record.wav", duration=DURATION, sr=SAMPLE_RATE):
    print(f"Recording for {duration} seconds...")
    audio = sd.rec(int(duration * sr), samplerate=sr, channels=1)
    sd.wait()
    sf.write(filename, audio, sr)
    print(f"Recording saved to {filename}")
    return filename

# ---------- Verification function (from before) ----------
def verify_speaker(file_path, model, le_user, df_profiles, device='cpu'):
    behavioral_feats = extract_behavioral_features(file_path)
    x_tensor = torch.tensor(behavioral_feats, dtype=torch.float32).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        output = model(x_tensor)
        pred_idx = output.argmax(dim=1).cpu().item()
        identified_speaker = le_user.inverse_transform([pred_idx])[0]
    
    stored_profile = df_profiles.loc[identified_speaker].values.reshape(1, -1)
    new_feat = behavioral_feats.reshape(1, -1)
    similarity_score = cosine_similarity(stored_profile, new_feat)[0][0]
    
    if similarity_score >= THRESHOLD:
        return identified_speaker, similarity_score
    else:
        return "No existing user", similarity_score

# ---------- Run Test ----------
if __name__ == "__main__":
    # Record new voice
    test_file = record_audio(duration=DURATION)
    
    # Identify speaker
    speaker, similarity = verify_speaker(test_file, model, le_user, df_profiles, device)
    
    print(f"Identified Speaker: {speaker}")
    print(f"Behavioral similarity with stored profile: {similarity:.4f}")


Recording for 5 seconds...
Recording saved to temp_record.wav
Identified Speaker: U007
Behavioral similarity with stored profile: 0.9873


In [86]:
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity
from scipy.special import softmax
import pandas as pd

def debug_verify(file_path, model, le_user, df_profiles, device='cpu', topk=5):
    # 1) extraction
    feats = extract_behavioral_features(file_path)
    print("-> Extracted feature vector shape:", feats.shape)
    
    # 2) feature/profile dim check
    profile_dim = df_profiles.shape[1]
    print("-> Profiles DataFrame shape:", df_profiles.shape)
    if feats.shape[0] != profile_dim:
        print("!!! Dimension mismatch: feature length != profile columns.")
        print(f"feature length = {feats.shape[0]}, profile columns = {profile_dim}")
        return
    
    # 3) model forward & softmax
    x_tensor = torch.tensor(feats, dtype=torch.float32).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        out = model(x_tensor)                 # raw logits
        logits = out.cpu().numpy().squeeze()
        probs = softmax(logits)
        pred_idx = int(np.argmax(logits))
    print("-> Logits shape:", logits.shape)
    print("-> Top 5 logits/probs:")
    for i in np.argsort(logits)[-5:][::-1]:
        user = le_user.inverse_transform([i])[0]
        print(f"   idx={i} user={user}  logit={logits[i]:.4f} prob={probs[i]:.4f}")
    pred_user = le_user.inverse_transform([pred_idx])[0]
    print("-> Model predicted user:", pred_user, "(index", pred_idx, ")")
    
    # 4) cosine similarities with all stored profiles
    profiles = df_profiles.values  # shape (N_users, dim)
    sims = cosine_similarity(profiles, feats.reshape(1, -1)).squeeze()
    # map indices to users
    users = df_profiles.index.tolist()
    top_idx = np.argsort(sims)[::-1][:topk]
    print(f"-> Top {topk} cosine similarities:")
    for idx in top_idx:
        print(f"   user={users[idx]}  similarity={sims[idx]:.4f}")
    # similarity of model-predicted user
    pred_user_idx = list(df_profiles.index).index(pred_user) if pred_user in df_profiles.index else None
    if pred_user_idx is not None:
        pred_sim = sims[pred_user_idx]
        print(f"-> Similarity with model-predicted user ({pred_user}): {pred_sim:.4f}")
    else:
        print("-> Predicted user not found in df_profiles index (label mismatch).")
    
    # 5) per-feature difference vs top-matched profile
    best_idx = top_idx[0]
    top_profile = profiles[best_idx]
    diff = feats - top_profile
    print("-> Per-feature summary vs top-matched profile:")
    print("   mean diff:", diff.mean(), " std diff:", diff.std(), " max abs diff:", np.max(np.abs(diff)))
    
    # 6) sanity checks
    print("-> Are any NaNs in features?", np.isnan(feats).any())
    print("-> Feature min/max:", feats.min(), feats.max())
    print("-> Profile min/max:", profiles.min(), profiles.max())
    
    # Return structured data
    return {
        "feats_shape": feats.shape,
        "model_logits": logits,
        "model_probs": probs,
        "model_pred_idx": pred_idx,
        "top_similarities": [(users[i], float(sims[i])) for i in top_idx],
        "pred_user_similarity": float(pred_sim) if pred_user_idx is not None else None
    }

# Example usage:
info = debug_verify("User_Voice/U004/01.wav", model, le_user, df_profiles, device)


-> Extracted feature vector shape: (26,)
-> Profiles DataFrame shape: (23, 26)
-> Logits shape: (23,)
-> Top 5 logits/probs:
   idx=3 user=U004  logit=13.9909 prob=0.9990
   idx=17 user=U018  logit=6.3452 prob=0.0005
   idx=9 user=U010  logit=5.8369 prob=0.0003
   idx=12 user=U013  logit=5.3250 prob=0.0002
   idx=2 user=U003  logit=4.3558 prob=0.0001
-> Model predicted user: U004 (index 3 )
-> Top 5 cosine similarities:
   user=U004  similarity=0.9914
   user=U016  similarity=0.9499
   user=U013  similarity=0.9493
   user=U003  similarity=0.9404
   user=U010  similarity=0.9403
-> Similarity with model-predicted user (U004): 0.9914
-> Per-feature summary vs top-matched profile:
   mean diff: 2.103437562355824  std diff: 8.117908051106363  max abs diff: 33.5062225341797
-> Are any NaNs in features? False
-> Feature min/max: -171.6705322265625 178.81856337655626
-> Profile min/max: -303.5700622558594 281.96848518936895


In [100]:
import os
import numpy as np
import sounddevice as sd
import soundfile as sf
import torch
import librosa
import parselmouth
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# ---------------- Configuration ----------------
SAMPLE_RATE = 16000
CLIP_DURATION = 10  # longer clips for stability
NUM_CLIPS = 2       # number of clips to average
THRESHOLD = 0.90    # similarity threshold
TEMP_FILE = "temp_live.wav"

# ---------------- Load Profiles ----------------
# df_profiles : DataFrame, index = user IDs, values = average behavioral features
# scaler : StandardScaler fitted on training dataset X

# ---------------- Behavioral Feature Extraction ----------------
def extract_behavioral_features(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
    y = y / np.max(np.abs(y)) if np.max(np.abs(y)) > 0 else y

    snd = parselmouth.Sound(file_path)
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]
    pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_std = np.std(pitch_values) if len(pitch_values) > 0 else 0

    point_process = parselmouth.praat.call(snd, "To PointProcess (periodic, cc)", 75, 500)
    jitter_local = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    shimmer_local = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)

    energy = np.mean(y**2)

    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    duration_sec = librosa.get_duration(y=y, sr=sr)
    speaking_rate = len(onsets)/duration_sec if duration_sec > 0 else 0

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    mfcc_mean = np.mean(mfccs.T, axis=0)

    features = np.hstack([mfcc_mean, pitch_mean, pitch_std, jitter_local, shimmer_local, energy, speaking_rate])
    return features

# ---------------- Standardization ----------------
def standardize_features(features):
    return scaler.transform(features.reshape(1, -1))

# ---------------- Record Audio Clip ----------------
def record_clip(filename=TEMP_FILE, duration=CLIP_DURATION):
    print(f"Recording {duration} seconds...")
    audio = sd.rec(int(duration * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1)
    sd.wait()
    sf.write(filename, audio, SAMPLE_RATE)
    return filename

# ---------------- Live Speaker Verification ----------------
def verify_speaker_live_similarity(df_profiles, scaler, num_clips=NUM_CLIPS, threshold=THRESHOLD):
    all_features = []

    # Record multiple clips
    for i in range(num_clips):
        print(f"\n--- Clip {i+1}/{num_clips} ---")
        clip_file = record_clip()
        feats = extract_behavioral_features(clip_file)
        all_features.append(feats)

    # Average features
    avg_feats = np.mean(np.array(all_features), axis=0)
    avg_feats_std = standardize_features(avg_feats)

    # Compute similarity to all stored user profiles
    similarities = []
    for user_id in df_profiles.index:
        profile_feats = df_profiles.loc[user_id].values.reshape(1, -1)
        sim = cosine_similarity(profile_feats, avg_feats.reshape(1, -1))[0][0]
        similarities.append((user_id, sim))

    # Pick best matching user
    best_user, best_sim = max(similarities, key=lambda x: x[1])

    if best_sim >= threshold:
        return best_user, best_sim
    else:
        return "No existing user", best_sim

# ---------------- Run ----------------
if __name__ == "__main__":
    speaker, similarity = verify_speaker_live_similarity(df_profiles, scaler)
    print(f"\nIdentified Speaker: {speaker}")
    print(f"Behavioral similarity with stored profile: {similarity:.4f}")



--- Clip 1/2 ---
Recording 10 seconds...

--- Clip 2/2 ---
Recording 10 seconds...

Identified Speaker: U001
Behavioral similarity with stored profile: 0.9937
